# SFANet — Dataset Preprocessing on Google Colab

Convert Kaggle zip files into the layout expected by SFANet dataset loaders.

| Dataset | Kaggle input | SFANet expects |
|---------|--------------|----------------|
| FaceForensics++ | `.mp4` video folders | Face PNGs + `train/val/test_<c23>.pickle` |
| Celeb-DF v2 | `.mp4` folders + test list | `*/images/<video_id>/*.png` |
| WildDeepfake | `*.tar.gz` face sequences | `train/test` + `real.pickle` / `fake.pickle` |

**Start with demo mode** (`MAX_VIDEOS = 5`) to validate before full extraction.

In [ ]:
# --- EDIT THESE PATHS ---
DRIVE = "/content/drive/MyDrive"
ZIP_FFPP   = f"{DRIVE}/ff-c23.zip"      # your FF++ zip path
ZIP_CELEB  = f"{DRIVE}/celeb-df-v2.zip"           # your Celeb-DF zip path
ZIP_WILD   = f"{DRIVE}/wild-deepfake.zip"      # your WildDeepfake zip path

WORK = "/content/sfanet_data"   # working dir (fast local disk)
OUT  = f"{DRIVE}/sfanet_preprocessed"  # save final output to Drive

# Demo mode: set to 5 for quick test, None for full dataset
MAX_VIDEOS = 5
MAX_FRAMES = 16      # faces extracted per video
FRAME_STRIDE = 10    # process every Nth frame
COMPRESSION = "c23"  # FF++ quality: raw | c23 | c40

In [ ]:
"""

from google.colab import drive
drive.mount("/content/drive")

!pip install -q opencv-python-headless retina-face facenet-pytorch albumentations torch torchvision

"""

import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Option A: full SFANet folder on Drive (includes preprocessing/)
SFANET_DRIVE = f"{DRIVE}/SFANet"
if Path(SFANET_DRIVE, "preprocessing").exists():
    if Path("/content/SFANet").exists():
        shutil.rmtree("/content/SFANet")
    shutil.copytree(SFANET_DRIVE, "/content/SFANet")
    print("Using SFANet from Drive (with preprocessing/)")
else:
    # Option B: clone GitHub (model only — upload preprocessing/ separately)
    if not Path("/content/SFANet").exists():
        !git clone https://github.com/akshaymina/SFANet.git /content/SFANet
    if not Path("/content/SFANet/preprocessing").exists():
        raise FileNotFoundError(
            "Upload your local SFANet/preprocessing/ folder to Drive at "
            f"{SFANET_DRIVE}/preprocessing/  OR copy it into /content/SFANet/"
        )

sys.path.insert(0, "/content/SFANet/preprocessing")
Path(WORK).mkdir(parents=True, exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)
print("Ready.")

In [ ]:
def unzip_if_needed(zip_path, dest):
    dest = Path(dest)
    if dest.exists() and any(dest.iterdir()):
        print(f"Already extracted: {dest}")
        return dest
    print(f"Extracting {zip_path} -> {dest}")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    return dest

RAW_FFPP  = unzip_if_needed(ZIP_FFPP,  f"{WORK}/raw/ffpp")
RAW_CELEB = unzip_if_needed(ZIP_CELEB, f"{WORK}/raw/celebdf")
RAW_WILD  = unzip_if_needed(ZIP_WILD,  f"{WORK}/raw/wild")

# Print top-level folders so you can fix paths if layout differs
for name, p in [("FF++", RAW_FFPP), ("Celeb-DF", RAW_CELEB), ("Wild", RAW_WILD)]:
    print(f"\n{name} contents:", os.listdir(p)[:15])

## 1. FaceForensics++

**Raw layout (after unzip):**
```
FaceForensics++/
├── original_sequences/youtube/c23/videos/*.mp4
├── original_sequences/actors/c23/videos/*.mp4
└── manipulated_sequences/{Deepfakes,Face2Face,FaceSwap,NeuralTextures}/c23/videos/*.mp4
```

**Output layout:**
```
FF++/
├── original_sequences/youtube/c23/images/<video>/0000.png
├── train_c23.pickle   # list of (relative_path, label)
├── val_c23.pickle
└── test_c23.pickle
```

If your zip has a different root, set `FFPP_RAW` below.

In [ ]:
import subprocess

FFPP_RAW = RAW_FFPP
for candidate in [RAW_FFPP, f"{RAW_FFPP}/FaceForensics++", f"{RAW_FFPP}/FaceForensics"]:
    if Path(candidate, "original_sequences").exists():
        FFPP_RAW = candidate
        break
print("FF++ raw root:", FFPP_RAW)

FFPP_OUT = f"{WORK}/processed/ffpp"
cmd = [
    sys.executable, "/content/SFANet/preprocessing/preprocess_ffpp.py",
    "--raw_root", FFPP_RAW,
    "--out_root", FFPP_OUT,
    "--compression", COMPRESSION,
    "--frame_stride", str(FRAME_STRIDE),
    "--max_frames", str(MAX_FRAMES),
]
if MAX_VIDEOS:
    cmd += ["--max_videos", str(MAX_VIDEOS)]
subprocess.run(cmd, check=True)

subprocess.run([
    sys.executable, "/content/SFANet/preprocessing/verify_datasets.py",
    "--dataset", "ffpp", "--root", FFPP_OUT, "--compression", COMPRESSION,
], check=True)

## 2. Celeb-DF v2

**Raw layout:**
```
Celeb-DF/
├── YouTube-real/*.mp4
├── Celeb-real/*.mp4
├── Celeb-synthesis/*.mp4
└── List_of_testing_videos.txt
```

**Output layout:**
```
Celeb-DF/
├── YouTube-real/images/00170/0000.png
├── Celeb-real/images/id1_0007/0000.png
├── Celeb-synthesis/images/id1_id0_0007/0000.png
└── List_of_testing_videos.txt
```

In [ ]:
CELEB_RAW = RAW_CELEB
for candidate in [RAW_CELEB, f"{RAW_CELEB}/Celeb-DF", f"{RAW_CELEB}/Celeb-DF-v2"]:
    if Path(candidate, "YouTube-real").exists() or Path(candidate, "Celeb-real").exists():
        CELEB_RAW = candidate
        break
print("Celeb-DF raw root:", CELEB_RAW)

CELEB_OUT = f"{WORK}/processed/celebdf"
cmd = [
    sys.executable, "/content/SFANet/preprocessing/preprocess_celebdf.py",
    "--raw_root", CELEB_RAW,
    "--out_root", CELEB_OUT,
    "--frame_stride", str(FRAME_STRIDE),
    "--max_frames", str(MAX_FRAMES),
]
if MAX_VIDEOS:
    cmd += ["--max_videos", str(MAX_VIDEOS)]
subprocess.run(cmd, check=True)

subprocess.run([
    sys.executable, "/content/SFANet/preprocessing/verify_datasets.py",
    "--dataset", "celebdf", "--root", CELEB_OUT,
], check=True)

## 3. WildDeepfake

WildDeepfake ships **pre-cropped 224×224 face images** inside `.tar.gz` archives — no face detection needed.

**Raw layout:**
```
deepfake_in_the_wild/
├── real train/*.tar.gz   (or real_train/)
├── real test/*.tar.gz
├── fake train/*.tar.gz
└── fake test/*.tar.gz
```

**Output layout:**
```
WildDeepfake/
├── train/real/<seq>/1057.png
├── train/fake/<seq>/1057.png
├── train/real.pickle
├── train/fake.pickle
└── test/...
```

In [ ]:
WILD_RAW = RAW_WILD
for candidate in [RAW_WILD, f"{RAW_WILD}/deepfake_in_the_wild"]:
    if Path(candidate).exists():
        WILD_RAW = candidate
        break
print("WildDeepfake raw root:", WILD_RAW)

WILD_OUT = f"{WORK}/processed/wilddeepfake"
subprocess.run([
    sys.executable, "/content/SFANet/preprocessing/preprocess_wilddeepfake.py",
    "--raw_root", WILD_RAW,
    "--out_root", WILD_OUT,
], check=True)

subprocess.run([
    sys.executable, "/content/SFANet/preprocessing/verify_datasets.py",
    "--dataset", "wilddeepfake", "--root", WILD_OUT,
], check=True)

## 4. Copy to Google Drive & test SFANet loaders

In [ ]:
import shutil
for name, src in [
    ("FaceForensics++", FFPP_OUT),
    ("Celeb-DF", CELEB_OUT),
    ("WildDeepfake", WILD_OUT),
]:
    dst = f"{OUT}/{name}"
    if Path(src).exists():
        if Path(dst).exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Saved -> {dst}")

print("\nFinal paths for SFANet config:")
print(f"  FF++:         {OUT}/FaceForensics++")
print(f"  Celeb-DF:     {OUT}/Celeb-DF")
print(f"  WildDeepfake: {OUT}/WildDeepfake")

In [ ]:
# Quick loader smoke-test (after preprocessing succeeds)
import sys
sys.path.insert(0, "/content/SFANet")

from dataset import load_dataset

cfg_celeb = {
    "root": f"{OUT}/Celeb-DF",
    "split": "test",
    "balance": False,
    "transforms": [
        {"name": "Resize", "params": {"height": 299, "width": 299}},
        {"name": "Normalize", "params": {"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]}},
    ],
}

ds = load_dataset("CelebDF")(cfg_celeb)
print(f"Celeb-DF test samples: {len(ds)}")
path, label = ds[0]
print(f"Sample: label={label}, path={path}")